# Week 11 · Day 2 — From Demo to Product
## Composing Classic + Generative ML  *(the course finale notebook)*

Yesterday you *used* a language model. Today you make it **reliable** and you **compose** it with the
classic models you have owned since Week 8. Five short demos, each a keeper from the lecture:

1. **System roles + in-context learning** — steer a *frozen* model with a system prompt and a few examples.
2. **Structured JSON output** — get data your code can `json.loads()` — and validate it.
3. **Chain-of-thought** — watch answer-only fail and *step-by-step* succeed.
4. **The combo pattern** (the centerpiece) — a classic classifier **routes**, a generative model **writes**.
5. **RAG** — ground the model in real documents so it stops making things up.

> **The model.** We use **Qwen2.5-0.5B-Instruct** — the same small **instruction-tuned** model you met on
> Day 1. It's tiny (494M params, runs on a laptop) so it loads fast and runs offline — *and* because it's
> instruction-tuned, the prompt-engineering techniques **actually work** here, hands-on. It's also small
> enough that you'll watch it **occasionally slip** — a good, honest reminder that reliability scales with
> model size. Everything runs **locally, no API keys.**

## Setup

We load two small, local models: **Qwen2.5-0.5B-Instruct** (the generator) and the default
**`sentiment-analysis`** classifier (a Week-8-style DistilBERT). Model loads are wrapped so a download
failure prints a message instead of crashing the notebook.

In [ ]:
import re, json
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)  # reproducibility; we also use greedy decoding (do_sample=False) so outputs are deterministic
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Load a GENERATIVE model (Qwen instruct) and a CLASSIC classifier (sentiment). Fail gracefully if offline.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
qwen = qwen_tok = sentiment = None
try:
    qwen_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    # transformers 5.3 renamed torch_dtype -> dtype. float32 is safe on CPU/MPS.
    qwen = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(DEVICE)
    qwen.eval()
    print(f'Generative model ready: {MODEL_NAME} ({sum(p.numel() for p in qwen.parameters()):,} params)')
except Exception as e:
    print(f'Could not load {MODEL_NAME} (offline?): {e}')

try:
    sentiment = pipeline('sentiment-analysis')  # -> distilbert-base-uncased-finetuned-sst-2-english
    print('Classic classifier ready: sentiment-analysis (DistilBERT-SST2)')
except Exception as e:
    print(f'Could not load the sentiment classifier (offline?): {e}')

In [ ]:
def chat(system, user, max_new_tokens=120):
    """Send a system + user message to Qwen; return its reply (deterministic, greedy)."""
    if qwen is None:
        return '[generative model unavailable in this environment]'
    messages = [{'role': 'system', 'content': system},
                {'role': 'user', 'content': user}]
    # transformers 5.3: use return_dict=True so we get input_ids + attention_mask, then splat with **inputs
    inputs = qwen_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(DEVICE)
    with torch.no_grad():
        out = qwen.generate(**inputs, max_new_tokens=max_new_tokens,
                            do_sample=False, pad_token_id=qwen_tok.eos_token_id)
    return qwen_tok.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

## Demo 1 — System Roles + In-Context Learning

Two levers that steer a **frozen** model (no weights change):

- **The system message** sets the model's *role and rules* — set once, obeyed on every reply.
- **In-context examples** (few-shot) show it a *pattern to continue*.

Watch a bare prompt ramble, a strict system prompt lock the format, and 3 examples teach a brand-new
output format the model had no reason to invent.

In [ ]:
review = 'The checkout process was an absolute nightmare.'

# (a) BARE: a generic system prompt, no format rules -> the model rambles a whole paragraph
print('BARE prompt:')
print('  ', chat('You are a helpful assistant.', f"Classify the sentiment of this review: '{review}'", 50))

# (b) CONTROLLED: the SYSTEM message sets the role AND the exact output format
# TODO 1: write a SYSTEM prompt that (a) gives the model the ROLE of a sentiment classifier and
#         (b) forces the exact FORMAT: exactly one uppercase word, POSITIVE or NEGATIVE, nothing else.
CLASSIFIER_SYSTEM = ____   # <- your role + strict-format system message
print('\nCONTROLLED (role + strict format):')
print('  ', repr(chat(CLASSIFIER_SYSTEM, review, 5)))

# (c) IN-CONTEXT LEARNING: teach a brand-new output format ([POS]/[NEG]) from 3 examples
def tag_prompt(new_review):
    return ("'Great product, love it!' -> [POS]\n"
            "'Broke instantly, awful.' -> [NEG]\n"
            "'Best purchase this year.' -> [POS]\n"
            f"'{new_review}' ->")

print('\nIN-CONTEXT (the model copies the [POS]/[NEG] format it was never told about):')
for r in ['Waste of money, very disappointed.', 'So happy with this, wonderful!', 'It stopped working after a week.']:
    print(f'   {r:38} -> {chat("Follow the pattern in the examples exactly.", tag_prompt(r), 6)!r}')

**Keeper #1:** few-shot doesn't *retrain* the model — it **steers a frozen model.** The system message
steers its *role and format*; the examples steer it to *continue a pattern* (here, a custom `[POS]/[NEG]`
tag it invented from three examples). Nothing was trained — you arranged the context so the answer is the
most probable continuation.

## Demo 2 — Structured JSON Output (make the model talk to your code)

A *word* is for a human. **Data** is for your program. We ask for JSON, then `json.loads()` it into a real
dict and use a field — hands-on. Because you can **never** trust raw output, we parse **defensively**
(strip code fences, find the `{...}`, fail gracefully).

> **Honest slip to watch for:** this 0.5B model usually returns valid JSON, but it may keep the price as the
> *string* `"$79"` instead of the *number* `79` — a small-model detail slip. That's exactly why you
> validate, and why reliable structured output at scale wants a bigger model (or constrained decoding).

In [ ]:
raw = chat('You extract fields from text and return ONLY a JSON object, no extra words.',
           'Extract product and price from: "I ordered the Acme Blender X2 for $79." Keys: product, price.', 60)
print('Qwen raw output:', repr(raw))

def parse_json(text):
    """Defensively pull a JSON object out of model text. Returns a dict, or None on failure."""
    text = re.sub(r'```(json)?|```', '', text).strip()   # strip any code fences
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

data = parse_json(raw)
if data is None:
    print('Parse FAILED -> in production you RETRY or fall back.')
else:
    print('Parsed into a Python dict your code can use:', data)
    print('  e.g. data["product"] =', data.get('product'), '| data["price"] =', data.get('price'))

Prompt to *increase the odds* of clean structure; engineer the **parse-and-validate (and retry)** to handle
the times it slips (like a string price). Belt and suspenders — that's the difference between a demo and a product.

## Demo 3 — Chain-of-Thought (let it think out loud)

The model writes **one token at a time**, and each token becomes context for the next. Force an instant
answer and it *blurts*; let it write the steps first and each step feeds the next — the generated words
are its **scratch paper.** Watch answer-only get it wrong, and step-by-step get it right.

> The big payoff of chain-of-thought grows with model size, but even this 0.5B model shows the effect clearly.

In [ ]:
q = 'A jacket is on sale for 20% off and now costs $40. What was the original price?'

print('ANSWER-ONLY :', repr(chat('Give only the final numeric answer, e.g. $NN.', q, 12)))
print('\nSTEP-BY-STEP:')
print(chat('Solve this step by step, then give the final answer.', q, 200))

Answer-only jumps straight to a wrong number; "think step by step" makes the model set up `0.8 · P = 40`
and solve `P = 50` — the correct original price. Same frozen model; you just gave it permission to use scratch paper.

## Demo 4 — The Combo Pattern  *(the centerpiece)*

The punchline of the whole course. You own two families of model:

- **Classic** (Weeks 1–9): reliable, cheap, fast, narrow — great at *deciding*.
- **Generative** (Weeks 10–11): flexible, creative — great at *writing*.

Don't choose. **Compose.** A classic sentiment classifier **routes** each review (like a triage nurse);
the generative model **writes** the tailored reply (the specialist). Three real reviews, end to end.

In [ ]:
reviews = [
    'This blender is amazing, it completely changed my morning routine!',
    'The item arrived cracked and customer support never replied.',
    'Fast shipping and it works exactly as described.',
]
SUPPORT_SYSTEM = ('You are a friendly customer-support agent for a kitchenware store. '
                  'Write a brief, warm reply of 1-2 sentences.')

def handle_review(text):
    # 1) CLASSIC model ROUTES: positive or negative?  (fast, reliable, near-free)
    if sentiment is None:
        return 'UNKNOWN', 0.0, '[classifier unavailable]'
    result = sentiment(text)[0]
    label, score = result['label'], result['score']
    # 2) GENERATIVE model WRITES: draft a reply matching the branch
    # TODO 2: branch on the classifier's decision. If NEGATIVE, build a `user` message asking for a
    #         short APOLOGY; otherwise ask for a short THANK-YOU. Reference the review via `text`.
    if label == 'NEGATIVE':
        user = ____   # <- ask the model to apologize for this negative review
    else:
        user = ____   # <- ask the model to thank the customer for this positive review
    return label, score, chat(SUPPORT_SYSTEM, user, 80)

for rev in reviews:
    label, score, reply = handle_review(rev)
    print(f'\nREVIEW : {rev}')
    print(f'  ROUTE (classic model) : {label}  ({score:.0%} confident)')
    print(f'  DRAFT (generative)    : {reply}')

**Keeper #3 — the keeper of the finale:** real-world ML is **composition**. Classic models *route and
decide*; generative models *write and create*. The classifier routes all three correctly; Qwen writes
clean, on-brand replies (a real apology for the cracked item, warm thank-yous for the happy ones). That's
exactly the shape of a real support-automation product — and you just built one.

## Demo 5 — RAG: Give the Model the Open Book

Hallucination is **intrinsic** — the model predicts *probable* tokens, not *true* ones. The fix isn't a
better prompt; it's **grounding**: retrieve the real document, put it in the prompt, *then* generate.
Closed-book (recall-and-bluff) becomes open-book (read-then-answer).

Our retriever is deliberately trivial — pick the document that shares the most words with the question.
(Real RAG uses embeddings, like Week 10's meaning-space — keyword overlap makes the *idea* obvious, and,
as you'll see, exposes RAG's weak link.)

In [ ]:
DOCS = [
    'The Acme Blender X2 has a 2-year warranty covering motor defects.',
    'Store hours are 9am to 6pm Monday through Friday, closed weekends.',
    'Returns are accepted within 30 days with the original receipt.',
    'The Acme Blender X2 ships with a tamper, a recipe book, and two jars.',
]

def retrieve(question, documents):
    """Toy retriever: pick the doc that shares the most words with the question."""
    q_words = set(re.findall(r'[a-z0-9]+', question.lower()))
    # TODO 3: score each document by how many words it shares with the question
    #         (compute the size of the overlap between q_words and each doc's word set).
    scores = [____ for d in documents]   # <- overlap count per document
    best = int(np.argmax(scores))
    return best, scores[best], documents[best]

def answer(question, grounded):
    if grounded:
        idx, score, doc = retrieve(question, DOCS)
        # TODO 4: build the GROUNDED call — pass the retrieved `doc` as context and the `question`.
        #         Use the system message below; assemble the user message from `doc` and `question`.
        user_msg = ____   # <- e.g. f'Context: {doc}\nQuestion: {question}'
        return chat('Answer using ONLY the provided context, in one short sentence.', user_msg, 40)
    # UNGROUNDED: no context -> the model recalls (and may bluff) from its weights
    return chat('Answer the question in one short sentence.', question, 40)

for q in ['What are the store hours?',
          'How long is the warranty on the Acme Blender X2?',
          'What is the return policy?']:
    idx, score, doc = retrieve(q, DOCS)
    print(f'\nQUESTION   : {q}')
    print(f'  retrieved: "{doc}"  (word overlap = {score})')
    print(f'  UNGROUNDED: {answer(q, grounded=False)}')
    print(f'  GROUNDED  : {answer(q, grounded=True)}')

**Keeper #2:** RAG doesn't make the model *know* more — it turns a **closed-book exam into an open-book
one.** Ungrounded, Qwen invents store hours and says the warranty is "1 year"; grounded, it reads the real
hours and the real "two years" straight off the retrieved doc.

**The honest limit — RAG is only as good as its retrieval.** On the *return policy* question, the keyword
retriever grabs the **wrong** document ("return" doesn't match "Returns"), so the model faithfully — but
wrongly — answers about the *warranty*. Garbage retrieval in, confident garbage out. That's why production
RAG uses **embeddings** (Week 10's meaning-space), not keyword overlap — and why you still verify.

## Reflect (3 questions)

1. **In-context learning:** In Demo 1, no weights changed — so in what sense did the model "learn" the
   `[POS]/[NEG]` format from your examples? What does that tell you about what an LLM is really doing when
   it follows a prompt?

2. **Composition:** In the combo pattern (Demo 4), *why* use a classic classifier to route instead of just
   asking Qwen "is this positive or negative, then write a reply" in one prompt? Name at least two reasons
   (think: reliability, cost, speed, testability).

3. **Grounding:** RAG fixed the store-hours and warranty answers but failed on the return-policy question.
   *Which* step failed there — retrieval or generation? What would you change to catch a wrong answer
   before a customer sees it?

## You can now build ___  *(course finale)*

Three months ago this was a wall of intimidating math. Look at what you can do now:

- **Fine-tune** a pretrained image or text model on your own data (Weeks 9–10).
- **Classify** text, **extract** structured data from messy input, and **generate** text and images.
- **Prompt deliberately** — system roles, few-shot, structured output, chain-of-thought — and know *when
  the model will lie* and *how to ground it* so it doesn't.
- **Compose** a classic model and a generative model into **one pipeline** and ship something real:
  a support-reply drafter, a document-grounded Q&A bot, a classify-then-generate app.

Underneath all of it is the one idea from Week 1: *define a loss, compute gradients, update the weights,
repeat.* Linear regression and the language model you drove today are the **same move** at different scales.
The rest — attention, convolution, diffusion, retrieval, composition — are tools you can now **name and combine.**

**Your portfolio piece:** combine at least two of the models you built this term into one thing that's yours.
You know the pattern now — composition.

**Go build things.** 🚀